In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA L4


In [2]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece
!pip uninstall -y torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import torch
import gc

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

set_seed(42)

## Data

In [4]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [5]:
df = pd.read_csv("train.csv")
df = df[["id", "input", "output"]]

df = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(df.shape)

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(train_df.shape, val_df.shape)

(5000, 3)
(4500, 3) (500, 3)


## Tokenizer

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary Size:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)
print("PAD Token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Vocabulary Size: 151643
EOS Token: <|im_end|>
PAD Token: <|endoftext|>


In [7]:
SYSTEM_PROMPT = (
    "আপনি একজন অভিজ্ঞ ও সহানুভূতিশীল বাংলা চিকিৎসক। "
    "রোগীর প্রশ্ন মনোযোগ দিয়ে বিশ্লেষণ করুন এবং প্রাসঙ্গিক চিকিৎসা জ্ঞান ব্যবহার করে যুক্তিসংগত, নির্ভুল ও সহজ ভাষায় উত্তর দিন। "
    "প্রথমে রোগীর সমস্যার সম্ভাব্য কারণ বা প্রেক্ষাপট সংক্ষেপে ব্যাখ্যা করুন, তারপর প্রয়োজনীয় পরামর্শ দিন। "
    "অপ্রয়োজনীয় তথ্য, পুনরাবৃত্তি বা অনুমানভিত্তিক দাবি করবেন না। "
    "উত্তর হবে স্বাভাবিক, আশ্বস্তকারী, পেশাদার এবং রোগী-বান্ধব বাংলায়।"
)
MAX_LENGTH = 768

## Format + Tokenize + Label masking (single pass)

Assistant-er response tokens-er upor loss lagano hocche (prompt part -100 diye mask kora).

In [8]:
def format_and_tokenize(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
    ]
    full_messages = prompt_messages + [
        {"role": "assistant", "content": example["output"]}
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        full_messages, tokenize=False, add_generation_prompt=False
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(
        full_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False
    )["input_ids"]

    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = full_ids.copy()
    for i in range(prompt_len):
        labels[i] = -100

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(format_and_tokenize, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_and_tokenize, remove_columns=val_dataset.column_names)

print(train_dataset)
print(train_dataset.column_names)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4500
})
['input_ids', 'attention_mask', 'labels']


In [10]:
def has_supervised_tokens(example):
    return any(l != -100 for l in example["labels"])

before = len(train_dataset)
train_dataset = train_dataset.filter(has_supervised_tokens)
val_dataset = val_dataset.filter(has_supervised_tokens)
after = len(train_dataset)

print(f"Train: {before} -> {after} (dropped {before - after} fully-truncated examples)")

Filter:   0%|          | 0/4500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Train: 4500 -> 925 (dropped 3575 fully-truncated examples)


In [11]:
lengths = [len(x) for x in train_dataset["input_ids"]]
print("Shortest:", min(lengths))
print("Average :", sum(lengths) / len(lengths))
print("Longest :", max(lengths))

Shortest: 557
Average : 767.5837837837838
Longest : 768


## Data collator

Custom collator jeta labels-o pad kore -100 diye. `DataCollatorForLanguageModeling` use kora jabe na, oita label masking overwrite kore fele.

In [12]:
def custom_data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id

    input_ids, attention_mask, labels = [], [], []
    for f in features:
        n_pad = max_len - len(f["input_ids"])
        input_ids.append(f["input_ids"] + [pad_id] * n_pad)
        attention_mask.append(f["attention_mask"] + [0] * n_pad)
        labels.append(f["labels"] + [-100] * n_pad)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

data_collator = custom_data_collator

## Model (QLoRA)

In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print(type(model))
print(model.device)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
cuda:0


In [14]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Training args

In [15]:
import transformers
print("Transformers version:", transformers.__version__)

_common_kwargs = dict(
    output_dir="./qwen_medical_bengali",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_strategy="steps",
    logging_steps=20,
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

try:
    training_args = TrainingArguments(eval_strategy="steps", **_common_kwargs)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="steps", **_common_kwargs)

print(training_args)

Transformers version: 5.13.1
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=2000,
eval_strategy=

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [17]:
print("Allocated:", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")
print("Reserved :", round(torch.cuda.memory_reserved()/1024**3, 2), "GB")

Allocated: 2.62 GB
Reserved : 3.26 GB


In [18]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
58,0.839846,0.774648


TrainOutput(global_step=58, training_loss=0.9608264462701206, metrics={'train_runtime': 890.7561, 'train_samples_per_second': 1.038, 'train_steps_per_second': 0.065, 'total_flos': 1.19547845148672e+16, 'train_loss': 0.9608264462701206, 'epoch': 1.0})

## Save + cleanup

In [19]:
trainer.save_model("./qwen_medical_bengali_final")
tokenizer.save_pretrained("./qwen_medical_bengali_final")

del trainer, model
gc.collect()
torch.cuda.empty_cache()

print("Model saved, memory cleared!")

Model saved, memory cleared!


## Inference (merged model - fast)

In [20]:
tokenizer = AutoTokenizer.from_pretrained("./qwen_medical_bengali_final")
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, "./qwen_medical_bengali_final")
model = model.merge_and_unload()
model.config.use_cache = True
model.eval()

print("Free GPU memory:", torch.cuda.mem_get_info()[0] / 1024**3, "GB")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Free GPU memory: 14.2686767578125 GB


In [21]:
from tqdm.auto import tqdm

def build_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(questions, batch_size=16):
    predictions = []
    pbar = tqdm(total=len(questions), desc="Generating")
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        prompts = [build_prompt(q) for q in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_LENGTH,
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        input_len = inputs.input_ids.shape[1]
        for out in outputs:
            predictions.append(tokenizer.decode(out[input_len:], skip_special_tokens=True).strip())

        pbar.update(len(batch))
    pbar.close()
    return predictions

In [22]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv


In [23]:
test_df = pd.read_csv("test.csv")
print(test_df.shape)
test_df.head()

(1000, 2)


,id,input
0,34654,"আমার লাইপেজ লেভেল ১৫৮, যা বেশি। অ্যামাইলেজ লেভ..."
1,3116,"তিন সপ্তাহ আগে আমার দাঁতে খুব ব্যথা ছিল, ৮ দিন..."
2,59478,"আমার স্বামীর বয়স ৫৬ বছর, তিনি খুব একটা মিশুক ন..."
3,56450,"হেলো, আমার বয়স ৩৮ বছর। আমার বাম পায়ে, একই দিকে..."
4,86380,গত মঙ্গলবার আমি ফ্লু শট এবং টিবি পরীক্ষা করিয়ে...


In [24]:
predictions = generate_batch(test_df["input"].tolist(), batch_size=16)

Generating:   0%|          | 0/1000 [00:00<?, ?it/s]

In [27]:
submission = pd.DataFrame({"id": test_df["id"], "output": predictions})

# empty string / NaN thakle fallback bosao — Kaggle null accept kore na
submission["output"] = submission["output"].fillna("").astype(str).str.strip()
empty_count = (submission["output"] == "").sum()
print(f"Empty predictions found: {empty_count}")

submission.loc[submission["output"] == "", "output"] = "দুঃখিত, উত্তর তৈরি করা সম্ভব হয়নি।"

submission.to_csv("submission.csv", index=False, encoding="utf-8")
print("Submission Saved!")

Empty predictions found: 3
Submission Saved!


In [30]:
from tqdm.auto import tqdm

def build_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(questions, batch_size=16):
    predictions = []
    pbar = tqdm(total=len(questions), desc="Generating")
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        prompts = [build_prompt(q) for q in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                repetition_penalty=1.3,
                no_repeat_ngram_size=3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        input_len = inputs.input_ids.shape[1]
        for out in outputs:
            text = tokenizer.decode(out[input_len:], skip_special_tokens=True)
            predictions.append(text.strip())

        pbar.update(len(batch))
    pbar.close()
    return predictions

In [31]:
predictions = generate_batch(test_df["input"].tolist(), batch_size=16)

Generating:   0%|          | 0/1000 [00:00<?, ?it/s]

In [32]:
submission = pd.DataFrame({"id": test_df["id"], "output": predictions})
submission.to_csv("submission.csv", index=False, encoding="utf-8")
print("Submission Saved!")

Submission Saved!


In [33]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>